# 2-clean&filter

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/interim/extract_16.csv", low_memory=False, dtype={"ID_orateur": str}
)
df.shape

(337041, 27)

In [2]:
# TODO: passer au fichier fusionné

In [3]:
# Ne garder que le code style NORMAL
df = df[df["Code_style"] == "NORMAL"]

# Exclure les prises de parole de "Mme la présidente" et "M. le président"
# Role_debat n'est pas bien identifié, utiliser Nom_orateur
df = df[~df["Nom_orateur"].str.strip().isin(["M. le président", "Mme la présidente"])]

# # TODO: AVISER selon introduction 15ème législature
# # Pour le sous cas de la 16 législature : nettoyer le fichier qui n'est pas au bon endroit
# # = date de 2021
# df = df[df["UID"] != "CRSANR5L16S2021O1N144"]

# Garder une trace de la longueur des interventions brutes
df["len_dirtytext"] = df["Texte"].str.len()

# Stabiliser le ID_orateur pour etre au format AN (pour matcher données)
df["ID_orateur"] = "PA" + df["ID_orateur"]

# Changer les missing values pour non_précisé (majoritaire) dans Code_parole
df["Code_parole"] = df["Code_parole"].fillna("non_précisé")

df.shape

(219086, 28)

In [4]:
# aperçu des répartitions
df.groupby("Code_parole", dropna=False)["len_dirtytext"].describe()

,count,mean,std,min,25%,50%,75%,max
Code_parole,,,,,,,,
AVIS_COM_1_10,1.0,1278.000000,NaN,1278.0,1278.0,1278.0,1278.00,1278.0
AVIS_COM_1_20,10951.0,389.245457,416.340444,4.0,96.0,273.0,538.00,5395.0
AVIS_GVT_1_20,9954.0,418.768033,581.874887,4.0,20.0,212.0,581.75,6481.0
PAROLE_1_1,1.0,32.000000,NaN,32.0,32.0,32.0,32.00,32.0
PAROLE_1_2,70643.0,819.476325,1053.234730,3.0,221.0,464.0,1000.00,21406.0
Raccroche_apres_inter,1.0,83.000000,NaN,83.0,83.0,83.0,83.00,83.0
non_précisé,127533.0,270.837924,575.767019,3.0,19.0,39.0,242.00,16991.0


In [5]:
# TODO: regrouper les interventions interrompues ?

**?????????aviser pour regrouper les interventions interrompues ?????????**

## Match députés

### Match infos générales (historique)

In [6]:
df_deputes = pd.read_csv("../data/raw/id-dep/deputes-historique(datan-datagouv).csv")
# suppression des colonnes non utiles qui introduisent soucis parsing
df_deputes = df_deputes.drop(columns=["mail", "twitter", "facebook", "website"])


In [7]:
print("shape avant fusion:", df.shape)

assert df_deputes["id"].is_unique, "ids du df_deputes non uniques !"

# Merge et virer la col id pour éviter doublon
df = df.merge(
    df_deputes,
    left_on="ID_orateur",
    right_on="id",
    how="left",
    suffixes=("", "_dep"),
    validate="many_to_one",  # check if merge keys are unique in right dataset
).drop(columns=["id"])  # supprimer la colonne id du df_deputes

print("shape après fusion:", df.shape)

shape avant fusion: (219086, 28)
shape après fusion: (219086, 50)


### Match temporel des affiliations

In [8]:
# recodage des grandes dénominations des groupes
# (moins sensible aux évolutions marginales de dénomination)

df_affiliation = pd.read_csv(
    "../data/raw/id-dep/datan_affiliations.csv", encoding="latin1", sep=";"
)  # format degeu

# Recoder les partis pour stabilité temporelle des noms
# TODO: visiblement d'autres : soc-a, agir-e,fi/lfi, UDI/modem, etc.
# Vérifier
recodage = {
    "RE": "REN",
    "LAREM": "REN",
    "DEM": "MODEM",
    "SOC": "PS",
    "NG": "PS",
    "LFI-NUPES": "LFI",
    "UDI-AGIR": "UDI",
    "UDI-A-I": "UDI",
    "LC": "UDI",
    "UDI_I": "UDI",
    "UDI-I": "UDI",
    "ECOLO": "ECO",
    "GDR-NUPES": "PCF",
    "GDR": "PCF",
    "LT": "LIOT",
    # Garde pour trace mais pas nécessaire car pas de changement
    # "LIOT": "LIOT",
    # "LR": "LR",
    # "RN": "RN",
    # "MODEM": "MODEM",
    # "LFI": "LFI",
    # "HOR": "HOR",
}

df_affiliation["libelleAbrev"] = df_affiliation["libelleAbrev"].astype(str).str.strip()
df_affiliation["parti_recod"] = df_affiliation["libelleAbrev"].replace(recodage)

In [9]:
# Si besoin de réexplorer les répartitions :

# df_affiliation["libelleAbrev"].value_counts()
# counts_abrev = df_affiliation["libelleAbrev"].value_counts()
# counts_abrege = df_affiliation["libelleAbrege"].value_counts()
# counts_libelle = df_affiliation["libelle"].value_counts()
# counts_recod = df_affiliation["parti_recod"].value_counts()

# counts_df = (
#     pd.DataFrame(
#         {
#             "libelleAbrev_count": counts_abrev,
#             "libelleAbrege_count": counts_abrege,
#             "libelle_count": counts_libelle,
#             "libelle_recod_count": counts_recod,
#         }
#     )
#     .fillna(0)
#     .astype(int)
# )

# # counts_df.to_csv("group_counts.csv")
# counts_df

In [10]:
##############################################################
# JE GARDE LE TEMPS QUE MATTHIAS PUISSE VOIR MA MAUVAISE IDÉE
# PASSÉ À UN LOOKUP ET RENVOI AFFILIATION TEMPORELLE VALIDE
##############################################################

# # EN cours : créer une fonction de renvoi du parti dans le temps
# # Et donc galérer avec les dates d'intervention vs date de début et fin affiliation ?

# # TODO: trouver ce qui merde car pour l'instant fait avec les pieds
# # pistes : cas des sans id_orateur ? cas des interv sans date ?
# # souci : tous ceux qui ont pas d'ID député > pas de renvoi de date début ou fin, etc.
# # Changer la logique ? > au final pas gros fichier
# # On peut partir sur lookup


# print("shape avant merge: ", df.shape)
# # Garder uniquement les colonnes utiles
# df_affiliation = df_affiliation[["mpId", "dateDebut", "dateFin", "parti_recod"]].copy()

# # Conversion des dates affiliation en datetime
# df_affiliation["dateDebut"] = pd.to_datetime(
#     df_affiliation["dateDebut"], errors="raise"
# )
# df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")


# # Conversion date intervention
# df["DateSeance_ts"] = pd.to_datetime(df["DateSeance"], format="%Y%m%d%H%M%S%f")

# # merge sur l'identifiant (ID_orateur vs mpId)
# df_merged = df.merge(df_affiliation, left_on="ID_orateur", right_on="mpId", how="left")

# # Garder uniquement les affiliations valides à la date de l’intervention
# df_match_affiliation = df_merged[
#     (df_merged["DateSeance_ts"] >= df_merged["dateDebut"])
#     & (df_merged["DateSeance_ts"] <= df_merged["dateFin"])
# ]

# print("shape après merge:", df_match_affiliation.shape)

In [11]:
# missing_ids = set(df["ID_orateur"]) - set(df_affiliation["mpId"])
# print(len(missing_ids), "orateurs n'ont aucune affiliation connue")

In [12]:
# mask = ~(
#     (df_merged["DateSeance_ts"] >= df_merged["dateDebut"]) &
#     (df_merged["DateSeance_ts"] <= df_merged["dateFin"])
# )
# print("interventions hors période:", mask.sum())

In [13]:
# Plutôt qu'un merge foireux parti sur un lookup ligne‑à‑ligne
# (= pb des orateurs non députés qui étaient pas présents, etc.)
# Le fichier est suffisamment réduit pour que le surplus de calcul soit pas un pb


# préparation des dates
df["DateSeance_ts"] = pd.to_datetime(
    df["DateSeance"], format="%Y%m%d%H%M%S%f", errors="raise"
)
df_affiliation["dateDebut"] = pd.to_datetime(
    df_affiliation["dateDebut"], errors="raise"
)
df_affiliation["dateFin"] = pd.to_datetime(df_affiliation["dateFin"], errors="raise")
# # aviser si jamais besoin traiter affiliations en cours
# df_affiliation["dateFin"] = df_affiliation["dateFin"].fillna(pd.Timestamp("2100-01-01"))

# indexer par mpId pour lookup rapide
aff_by_mp = {
    mp: g[["dateDebut", "dateFin", "parti_recod"]].to_dict("records")
    for mp, g in df_affiliation.groupby("mpId")
}


def get_parti_for_row(row):
    mp = row.get("ID_orateur")  # correspond au mpId
    # gérer le cas des orateurs non députés
    if pd.isna(mp) or mp not in aff_by_mp:
        return None
    # récupérer le ts de l'intervention
    ts = row.get("DateSeance_ts")
    if pd.isna(ts):
        return None
    # retourner l'affiliation qui colle à la date d'intervention
    for rec in aff_by_mp[mp]:
        if rec["dateDebut"] <= ts <= rec["dateFin"]:
            return rec["parti_recod"]
    return None


# appliquer et marquer les inconnus (pour repérage futur)
df["parti_affiliation"] = df.apply(get_parti_for_row, axis=1)
df["parti_affiliation"] = df["parti_affiliation"].fillna("UNKNOWN")

print(
    "affectés :",
    df["parti_affiliation"].ne("UNKNOWN").sum(),
    "UNKNOWN :",
    (df["parti_affiliation"] == "UNKNOWN").sum(),
)
# TODO: envisager de forcer le renvoi de la derniere affiliation connue de df_deputes ?
# Aviser pour des matchs plus précis (cas limites, etc.) sur la base des repérages de matthias.
# Aviser le cas des UNKNOWN = EPR et fusion REN ?

affectés : 181604 UNKNOWN : 37482


In [14]:
# # SI besoin de recomprendre la logique
# # de la récupération des affiliations :

# # type et aperçu
# for mp, g in df_affiliation.groupby("mpId"):
#     print(mp, type(g))
#     print(g.head())
#     break

# # ou inspecter le résultat stocké
# print(type(aff_by_mp["PA795864"]))  # list
# print(aff_by_mp["PA795864"][:2])  # 2 premiers enregistrements (dicts)

In [15]:
df["parti_affiliation"].value_counts().sort_index()

parti_affiliation
AGIR-E         3
ECO        14027
FI            69
HOR         5053
LFI        36484
LIOT        4031
LR         26508
MODEM      14408
NI          1713
PCF        10314
PS          6899
REN        36230
RN         21684
SOC-A       4179
UDI            2
UNKNOWN    37482
Name: count, dtype: int64

In [16]:
df["groupeAbrev"].value_counts().sort_index()

groupeAbrev
AGIR-E           3
DEM          17668
DR           19220
ECOLO          144
ECOS         15743
EPR          37006
FI              53
GDR           5837
GDR-NUPES     4433
HOR           6130
LAREM         1192
LES-REP        981
LFI-NFP      29733
LFI-NUPES     4537
LIOT          4287
LR            5483
NI            4575
RE           14894
RN           21617
SOC          10489
SOC-A          885
UDI_I            2
UDR            303
Name: count, dtype: int64

## Export

In [ ]:
# Export du csv nettoyé
df.to_csv("../data/interim/data_cleaning.csv", index=False)

# # NB: certaines col du df_deputes introduisent une erreur à l'import/export
# # Elles ne sont pas utilisées ici, mais si besoin de les utiliser
# # forcer le QUOTE_ALL permet de résoudre
# # (adresses et réseaux sociaux contenant saut de lignes = erreurs de parsing (cas eric.martineau))

# import csv  # pour utiliser csv.QUOTE_ALL et résoudre le soucis d'écart.
# df.to_csv(
#     "../data/interim/data_cleaning.csv",
#     index=False,
#     quoting=csv.QUOTE_ALL,  # permet de résoudre le soucis
# )

In [18]:
df.shape

(219086, 52)

In [19]:
# verif excriture/lecture ok
df_test = pd.read_csv(
    "../data/interim/data_cleaning.csv", low_memory=False, dtype={"ID_orateur": str}
)
df_test.shape

(219086, 52)

In [20]:
# TODO: regrouper les interventions interrompues ?